# Tool System V2 — Kaggle Bootstrap

Notebook bootstrap cho dự án độc lập Tool System V2. Mã nguồn bao gồm hạ tầng Chrome/noVNC và các nút nghiệp vụ Bước 05–07 được chuyển từ Money AI. Không đọc khóa hoặc dữ liệu đăng nhập từ Supabase cũ.

Sau khi chạy các cell kiểm thử, cell cuối sẽ hiển thị dashboard có nút Khởi động Chrome, liên kết Mở màn hình Chrome và các nút Bước 05–07.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/duykhanhlongchutayninh-debug/tool-system-v2.git"
WORKDIR = Path("/kaggle/working/tool-system-v2")

if not (WORKDIR / ".git").is_dir():
    if WORKDIR.exists():
        raise RuntimeError(f"Repository path exists but is not a Git checkout; preserve it and inspect: {WORKDIR}")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)], check=True)
else:
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", "main"], check=True)

print("Python:", sys.version)
print("Working directory:", WORKDIR)

In [ ]:
%cd /kaggle/working/tool-system-v2
!python -m pip install -e .
!python -m pip install -r requirements-runtime.txt
!python -m unittest discover -s tests -v

In [ ]:
import json
from urllib.request import urlopen
from urllib.error import URLError

CDP_VERSION_URL = "http://127.0.0.1:9222/json/version"
try:
    with urlopen(CDP_VERSION_URL, timeout=3) as response:
        if response.status != 200:
            raise RuntimeError(f"CDP returned HTTP {response.status}")
        data = json.loads(response.read().decode("utf-8"))
    if not data.get("Browser") or not str(data.get("webSocketDebuggerUrl", "")).startswith("ws"):
        raise RuntimeError("CDP response is missing required browser metadata")
    print(json.dumps({"status": "PASS", "browser": data["Browser"]}, ensure_ascii=False))
except (URLError, TimeoutError, OSError, RuntimeError, json.JSONDecodeError) as exc:
    print(json.dumps({"status": "NOT_VERIFIED", "reason": type(exc).__name__, "chrome_started": False}, ensure_ascii=False))
    print("Use the dashboard in the final cell to start Chrome; this probe itself does not launch it.")

## Giới hạn an toàn

- Chrome, noVNC, CDP và đường dẫn màn hình được kiểm tra độc lập.
- Bước 05/06/07 không đăng Facebook; bước 07 chỉ gửi video đã xác minh cho Telegram duyệt.
- Không chép token, cookies, khóa hay hồ sơ đăng nhập Money AI sang dự án mới.
- Kaggle session phải đang chạy; không có cam kết rằng notebook sẽ tự chạy khi bật CPU.
- Chỉ ghi nhận PASS khi cell hoặc nút dashboard trả bằng chứng thật.

In [ ]:
%cd /kaggle/working/tool-system-v2
from scripts.tool_system_dashboard import show_dashboard
show_dashboard()